<a href="https://colab.research.google.com/github/Shturman71/Colab/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22Karaokelight_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Комментармм** 1 Перед выполнеием основного кода запустить ячейку 8 для генерации файлов original_en.txt и translated_ru.txt из qa_data.csv загруженного вручную в папку content. Или сразу загрузить их в папку вручную

The notebook automatically saves your changes. To use it again with new data, you'll need to upload your new qa_data.csv file. Then, run the cell that converts the CSV into original_en.txt and translated_ru.txt (which is cell 44b8500f). After that, you can run this cell (jURZFdF8RDol) again to generate the karaoke video. I've added a clarifying comment to the main function in the current cell to serve as a reminder for this workflow.


In [13]:
try:
    import edge_tts
except ImportError:
    !pip install edge-tts
    import edge_tts

import asyncio
from pydub import AudioSegment
from moviepy.editor import TextClip, AudioFileClip, ColorClip, CompositeVideoClip, concatenate_videoclips
import os
import re
import shutil
from google.colab import files
import tempfile

# --- ImageMagick Installation and Configuration ---
# Install ImageMagick
print("Checking for ImageMagick installation...")
convert_path = shutil.which("convert")
if convert_path is None:
    print("ImageMagick not found. Installing...")
    !apt-get update -qq > /dev/null
    !apt-get install imagemagick -y -qq > /dev/null
    convert_path = shutil.which("convert") # Get path after installation
    print("ImageMagick installed.")
else:
    print("ImageMagick is already installed.")

# Configure MoviePy to use ImageMagick
import moviepy.config as mpy_config

# Explicitly set the environment variable for ImageMagick binary
# This often resolves 'unset' or 'no such file or directory' errors with TextClip
if convert_path:
    os.environ['IMAGEMAGICK_BINARY'] = convert_path
    print(f"Environment variable IMAGEMAGICK_BINARY set to: {os.environ['IMAGEMAGICK_BINARY']}")
    # Also inform MoviePy's internal config
    if mpy_config.get_setting("IMAGEMAGICK_BINARY") != convert_path:
        mpy_config.change_settings({"IMAGEMAGICK_BINARY": convert_path})
        print("MoviePy configured to use ImageMagick (internal setting).")
    else:
        print("MoviePy is already configured for ImageMagick (internal setting).")
else:
    print("WARNING: ImageMagick 'convert' executable not found, cannot set IMAGEMAGICK_BINARY.")

# Modify ImageMagick policy to allow temporary file creation (critical for TextClip)
policy_file = "/etc/ImageMagick-6/policy.xml"
print("Modifying ImageMagick policy...")
# These sed commands remove lines that restrict common operations often needed by TextClip
# Removing is often more effective than trying to change 'rights="none"' to 'rights="all"'
# or commenting, as some environments might still try to process commented-out lines or sed can be tricky.
!sudo sed -i '/<policy domain="coder" rights="none" pattern="PS" \/>/d' {policy_file}
!sudo sed -i '/<policy domain="coder" rights="none" pattern="PS2" \/>/d' {policy_file}
!sudo sed -i '/<policy domain="coder" rights="none" pattern="PS3" \/>/d' {policy_file}
!sudo sed -i '/<policy domain="coder" rights="none" pattern="EPS" \/>/d' {policy_file}
!sudo sed -i '/<policy domain="coder" rights="none" pattern="PDF" \/>/d' {policy_file}
!sudo sed -i '/<policy domain="coder" rights="none" pattern="XPS" \/>/d' {policy_file}
!sudo sed -i '/<policy domain="path" rights="none" pattern="@\*"\/>/d' {policy_file}

# Optional: a general replacement to allow read/write for coder/delegate domains if specific removals aren't enough
# It's better to be specific, but this is a fallback.
# !sudo sed -i 's/rights="none" pattern="\([A-Z]*\)"/rights="read|write" pattern="\1"/g' {policy_file}

print("ImageMagick policy modified. You may need to restart the runtime for full effect, but often not.")
print("--- ImageMagick setup complete ---")

# --- Helper Functions ---
def delete_directory_with_contents(directory_path):
    """Удаляет указанный каталог и все его содержимое."""
    if os.path.exists(directory_path):
        try:
            shutil.rmtree(directory_path)
            print(f"Каталог '{directory_path}' и все его содержимое успешно удалены.")
        except OSError as e:
            print(f"Ошибка: Не удалось удалить каталог {directory_path}. {e}")
    else:
        print(f"Каталог '{directory_path}' не существует.")

def create_silent_mp3(filename, duration_ms):
    """Создает пустой (бесшумный) MP3-файл указанной длительности."""
    silent_audio = AudioSegment.silent(duration=duration_ms)
    silent_audio.export(filename, format="mp3")
    print(f"Создан заглушечный (бесшумный) аудиофайл: {filename} ({duration_ms / 1000} сек)")

# --- НАСТРОЙКИ ---
# Голоса для озвучки
VOICE_RU = "ru-RU-DmitryNeural" # Дмитрий - русский голос
VOICE_EN = "en-US-GuyNeural"    # Гай - английский голос

OUTPUT_FOLDER = "simple_karaoke"
EN_FILE = "original_en.txt"       # Английский текст из источников [2-5] - Reverted filename
RU_FILE = "translated_ru.txt"     # Ваш перевод из источников [1, 6-13]
BACKGROUND_MUSIC_FILE = "background.mp3" # Файл фоновой музыки
PAUSE_DURATION = 10.0 # @param {type:"number"} # Длительность паузы между текстами в секундах. Измените это значение.
BACKGROUND_MUSIC_VOLUME = 0.1 # Громкость фоновой музыки (от 0.0 до 1.0)
GENERATE_AUDIO_LANGUAGES = "both" # @param ['ru', 'en', 'both']

# Выбор голоса для русского текста (вопроса)
VOICE_FOR_RUSSIAN_TEXT = "English Voice (Guy)" # @param ["Russian Voice (Dmitry)", "English Voice (Guy)"]
# Выбор голоса для английского текста (ответа)
VOICE_FOR_ENGLISH_TEXT = "English Voice (Guy)" # @param ["English Voice (Guy)", "Russian Voice (Dmitry)"]

# Новая настройка: Режим генерации финального видео
FINAL_VIDEO_GENERATION_MODE = "individual_clips_zip" # @param ["individual_clips_zip", "single_fast_ffmpeg"]





if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

async def generate_voice(text, voice, filename):
    """Синтез речи через edge-tts, с обработкой ошибок."""
    try:
        communicate = edge_tts.Communicate(text, voice)
        await communicate.save(filename)
        return filename # Return filename on success
    except Exception as e:
        print(f"WARNING: Error generating audio for text: '{text[:50]}...' with voice '{voice}'. Error: {e}")
        return None # Return None on failure

def create_text_video(text, audio_path, color, max_chars_per_line=40):
    """Создание видео-фрагмента: текст на однотонном фоне"""
    audio = AudioFileClip(audio_path)
    # Создаем фон (темно-синий, как космос, который видел Ли [4])
    bg = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(audio.duration)

    # Разделение текста на строки с ограничением по символам, включая очень длинные слова
    words = text.split()
    lines = []
    current_line = []
    current_len = 0

    for word in words:
        if not current_line: # Start of a new line
            current_line.append(word)
            current_len = len(word)
        elif current_len + 1 + len(word) <= max_chars_per_line: # Word fits with a space
            current_line.append(word)
            current_len += 1 + len(word)
        else: # Word does not fit, start a new line
            lines.append(' '.join(current_line))
            current_line = [word]
            current_len = len(word)

        # Handle cases where a single word is longer than max_chars_per_line
        while current_len > max_chars_per_line:
            if not current_line: # Should not happen if logic is correct
                break

            # Take a chunk and add to lines
            lines.append(current_line[0][:max_chars_per_line])
            current_line[0] = current_line[0][max_chars_per_line:] # Remaining part of the word
            current_len = len(current_line[0])

            if not current_line[0]: # If the word was fully consumed
                current_line = []
                current_len = 0
                break # Move to the next word

    if current_line:
        lines.append(' '.join(current_line))

    formatted_text = '\n'.join(lines)

    try:
        # Наложение текста
        txt = TextClip(formatted_text, fontsize=28, color=color, size=(1100, None),
                       method='caption', font='Arial').set_duration(audio.duration).set_position('center')
    except Exception as e:
        print(f"WARNING: Error creating TextClip for text: '{text}' (formatted: '{formatted_text[:100]}...') with fontsize={28}, max_chars_per_line={max_chars_per_line}. Error: {e}")
        # Return a blank clip to allow the script to continue
        return ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(audio.duration)
    return CompositeVideoClip([bg, txt]).set_audio(audio)

async def main():
    # --- Инструкции для повторного использования ---
    # Для повторного использования этого скрипта с новыми данными:
    # 1. Загрузите ваш новый 'qa_data.csv' в среду Colab.
    # 2. Запустите ячейку (с ID `44b8500f`), которая преобразует 'qa_data.csv' в 'original_en.txt' и 'translated_ru.txt'.
    # 3. После успешного создания текстовых файлов, вы можете повторно запустить эту ячейку (jURZFdF8RDol).
    # -------------------------------------------------

    # Проверка наличия ваших файлов в Colab
    if not os.path.exists(EN_FILE) or not os.path.exists(RU_FILE):
        print(f"Загрузите {EN_FILE} и {RU_FILE} в боковую панель!")
        return

    # Проверка наличия файла фоновой музыки
    if not os.path.exists(BACKGROUND_MUSIC_FILE):
        print(f"Загрузите {BACKGROUND_MUSIC_FILE} в боковую панель для фоновой музыки!")
        return

    # Загрузка фоновой музыки один раз
    background_audio_clip = AudioFileClip(BACKGROUND_MUSIC_FILE)

    # Улучшенное разбиение текста на предложения
    def get_sentences(file_path):
        sentences = []
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    sentences.append(line)
        return sentences

    en_sents = get_sentences(EN_FILE)
    ru_sents = get_sentences(RU_FILE)


    # Determine which actual voice to use based on user selection
    selected_voice_for_russian_text = VOICE_RU if VOICE_FOR_RUSSIAN_TEXT == "Russian Voice (Dmitry)" else VOICE_EN
    selected_voice_for_english_text = VOICE_EN if VOICE_FOR_ENGLISH_TEXT == "English Voice (Guy)" else VOICE_RU

    for i, (en, ru) in enumerate(zip(en_sents, ru_sents)):
        temp_ru = f"ru_{i}.mp3"
        temp_en = f"en_{i}.mp3"
        output_mp4 = f"{OUTPUT_FOLDER}/{i+1:03d}.mp4"

        # Determine which audio files will actually be generated
        audio_ru_will_be_generated = GENERATE_AUDIO_LANGUAGES in ["ru", "both"]
        audio_en_will_be_generated = GENERATE_AUDIO_LANGUAGES in ["en", "both"]

        # 1. Озвучка - Generate audio files
        audio_generation_tasks = []

        if audio_ru_will_be_generated:
            audio_generation_tasks.append(generate_voice(ru.strip(), selected_voice_for_russian_text, temp_ru))
        if audio_en_will_be_generated:
            audio_generation_tasks.append(generate_voice(en.strip(), selected_voice_for_english_text, temp_en))

        ru_audio_file_path = None
        en_audio_file_path = None

        if audio_generation_tasks:
            results = await asyncio.gather(*audio_generation_tasks)

            # Process results for RU audio
            if audio_ru_will_be_generated:
                # The index for RU audio result is 0 if it's the first in audio_generation_tasks
                # It would be 0 if only RU or if RU and EN (RU first)
                if results[0] is not None and os.path.exists(results[0]):
                    ru_audio_file_path = results[0]
                else:
                    print(f"WARNING: RU audio for segment {i+1} failed to generate. Creating silent placeholder.")
                    create_silent_mp3(temp_ru, int(PAUSE_DURATION * 1000)) # Use PAUSE_DURATION as a fallback
                    ru_audio_file_path = temp_ru

            # Process results for EN audio
            if audio_en_will_be_generated:
                # Adjust index based on whether RU audio was also attempted
                # If RU audio was also attempted, EN audio is at index 1
                en_result_idx = 0 if not audio_ru_will_be_generated else 1
                if results[en_result_idx] is not None and os.path.exists(results[en_result_idx]):
                    en_audio_file_path = results[en_result_idx]
                else:
                    print(f"WARNING: EN audio for segment {i+1} failed to generate. Creating silent placeholder.")
                    create_silent_mp3(temp_en, int(PAUSE_DURATION * 1000)) # Use PAUSE_DURATION as a fallback
                    en_audio_file_path = temp_en
        else:
            print(f"Предупреждение: Не выбрано ни одной опции для генерации аудио для пары {i+1}. Продолжаю без аудио для этой пары.")


        # Создаем клип для паузы с фоновой музыкой
        # Убедимся, что фоновая аудиозапись достаточно длинная или зациклена.
        # Для простоты возьмем сегмент фоновой аудиозаписи и уменьшим громкость.
        bg_music_segment = background_audio_clip.subclip(0, PAUSE_DURATION).volumex(BACKGROUND_MUSIC_VOLUME)
        pause_clip = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(PAUSE_DURATION)
        pause_clip = pause_clip.set_audio(bg_music_segment)

        # 2. Создание видео-частей и их сборка
        clips_to_assemble = []

        # Process Question part (RU text content)
        if ru_audio_file_path and os.path.exists(ru_audio_file_path):
            clip_question_video = create_text_video(ru.strip(), ru_audio_file_path, 'yellow', max_chars_per_line=40)
            clips_to_assemble.append(clip_question_video)
            clips_to_assemble.append(pause_clip) # Pause after the question

        # Process Answer part (EN text content)
        if en_audio_file_path and os.path.exists(en_audio_file_path):
            clip_answer_video = create_text_video(en.strip(), en_audio_file_path, 'cyan', max_chars_per_line=40)
            clips_to_assemble.append(clip_answer_video)
            clips_to_assemble.append(pause_clip) # Pause after the answer

        # Fallback if no audio/video segments were created for this pair
        if not clips_to_assemble:
            final_video = ColorClip(size=(1280, 720), color=(10, 10, 40)).set_duration(PAUSE_DURATION)
            print(f"WARNING: No video segments created for pair {i+1}. Using a placeholder ColorClip.")
        else:
            final_video = concatenate_videoclips(clips_to_assemble)

        final_video.write_videofile(output_mp4, fps=24, codec='libx264', audio_codec='aac')

        # Очистка временных аудио
        if os.path.exists(temp_ru):
            os.remove(temp_ru)
        if os.path.exists(temp_en):
            os.remove(temp_en)
        print(f"Видео {i+1:03d} создано.")

    print("\n--- Завершение процесса ---")

    if FINAL_VIDEO_GENERATION_MODE == "individual_clips_zip":
        # Шаг 3: Архивация и автоматическое скачивание отдельных клипов
        print(f"Архивирую отдельные видеоклипы из папки '{OUTPUT_FOLDER}'...")
        shutil.make_archive('karaoke_simple', 'zip', OUTPUT_FOLDER)
        print("Архив 'karaoke_simple.zip' готов к скачиванию.")
        files.download('karaoke_simple.zip')
        print("Скачивание архива 'karaoke_simple.zip' завершено.")

    elif FINAL_VIDEO_GENERATION_MODE == "single_fast_ffmpeg":
        final_video_output = "final_karaoke_video.mp4"
        concat_list_file = "files_to_concat.txt"

        # Получить список всех сгенерированных видеофайлов, отсортированных по имени
        video_files = sorted([f for f in os.listdir(OUTPUT_FOLDER) if f.endswith('.mp4')])

        if not video_files:
            print(f"Ошибка: В папке '{OUTPUT_FOLDER}' не найдено MP4-видеофайлов для объединения.")
        else:
            print(f"Объединяю {len(video_files)} видеофайлов в один с помощью ffmpeg...")

            # Создать текстовый файл со списком видео для ffmpeg concat demuxer
            with open(concat_list_file, 'w') as f:
                for video_file in video_files:
                    f.write(f"file '{os.path.join(OUTPUT_FOLDER, video_file)}'\n")

            # Команда ffmpeg для объединения видео
            command = f"ffmpeg -f concat -safe 0 -i {concat_list_file} -c copy {final_video_output}"

            # Выполнить команду ffmpeg
            !{command}

            print(f"Все видеофайлы объединены в '{final_video_output}'.")

            # Очистка временного файла списка
            if os.path.exists(concat_list_file):
                os.remove(concat_list_file)
            else:
                print(f"Предупреждение: Файл списка '{concat_list_file}' не найден для удаления.")


            # Очистка временных папок. If not needed for debugging, can be removed
            if os.path.exists(OUTPUT_FOLDER):
              shutil.rmtree(OUTPUT_FOLDER)
              print(f"Папка '{OUTPUT_FOLDER}' и ее содержимое удалены.")

            # Скачать объединенное видео
            if os.path.exists(final_video_output):
                files.download(final_video_output)
                print(f"Файл '{final_video_output}' готов к скачиванию.")
            else:
                print(f"Ошибка: Объединенный видеофайл '{final_video_output}' не был создан.")
    else:
        print(f"Неизвестный режим генерации финального видео: {FINAL_VIDEO_GENERATION_MODE}. Ничего не сгенегировано.")

# Запуск в Colab
await main()

Checking for ImageMagick installation...
ImageMagick is already installed.
Environment variable IMAGEMAGICK_BINARY set to: /usr/bin/convert
MoviePy is already configured for ImageMagick (internal setting).
Modifying ImageMagick policy...
ImageMagick policy modified. You may need to restart the runtime for full effect, but often not.
--- ImageMagick setup complete ---
Moviepy - Building video simple_karaoke/001.mp4.
MoviePy - Writing audio in 001TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video simple_karaoke/001.mp4



Moviepy - Done !
Moviepy - video ready simple_karaoke/001.mp4
Видео 001 создано.
Moviepy - Building video simple_karaoke/002.mp4.
MoviePy - Writing audio in 002TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video simple_karaoke/002.mp4



Moviepy - Done !
Moviepy - video ready simple_karaoke/002.mp4
Видео 002 создано.
Moviepy - Building video simple_karaoke/003.mp4.
MoviePy - Writing audio in 003TEMP_MPY_wvf_snd.mp4


...

MoviePy - Done.
Moviepy - Writing video simple_karaoke/005.mp4



Moviepy - Done !
Moviepy - video ready simple_karaoke/005.mp4
Видео 005 создано.
Moviepy - Building video simple_karaoke/006.mp4.
MoviePy - Writing audio in 006TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video simple_karaoke/006.mp4



t:  62%|██████▏   | 432/693 [00:09<00:10, 25.19it/s, now=None]

OSError: [Errno 32] Broken pipe

MoviePy error: FFMPEG encountered the following error while writing file simple_karaoke/006.mp4:

 b'[in#0/rawvideo @ 0x3dc67bc0] Error during demuxing: Immediate exit requested\n[rawvideo @ 0x3dc91c40] Invalid buffer size, packet size 65536 < expected frame_size 2764800\n[vist#0:0/rawvideo @ 0x3dc7ba40] [dec:rawvideo @ 0x3dc91180] Error submitting packet to decoder: Invalid argument\n[mov,mp4,m4a,3gp,3g2,mj2 @ 0x3dc7c9c0] stream 0, offset 0x4818e: partial file\n[in#1/mov,mp4,m4a,3gp,3g2,mj2 @ 0x3dc7c5c0] Error during demuxing: Immediate exit requested\n[out#0/mp4 @ 0x3dc7f840] Error writing trailer: Immediate exit requested\n[out#0/mp4 @ 0x3dc7f840] Error closing file: Immediate exit requested\n'

In [ ]:
policy_file = "/etc/ImageMagick-6/policy.xml"

print(f"Content of {policy_file}:")
with open(policy_file, 'r') as f:
    policy_content = f.read()
    print(policy_content)

if 'rights="all"' in policy_content:
    print("\nConfirmation: 'rights=\"all\"' found in ImageMagick policy.xml")
else:
    print("\nWarning: 'rights=\"all\"' NOT found in ImageMagick policy.xml. The policy might not have been applied correctly.")

Content of /etc/ImageMagick-6/policy.xml:
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE policymap [
  <!ELEMENT policymap (policy)*>
  <!ATTLIST policymap xmlns CDATA #FIXED ''>
  <!ELEMENT policy EMPTY>
  <!ATTLIST policy xmlns CDATA #FIXED '' domain NMTOKEN #REQUIRED
    name NMTOKEN #IMPLIED pattern CDATA #IMPLIED rights NMTOKEN #IMPLIED
    stealth NMTOKEN #IMPLIED value CDATA #IMPLIED>
]>
<!--
  Configure ImageMagick policies.

  Domains include system, delegate, coder, filter, path, or resource.

  Rights include none, read, write, execute and all.  Use | to combine them,
  for example: "read | write" to permit read from, or write to, a path.

  Use a glob expression as a pattern.

  Suppose we do not want users to process MPEG video images:

    <policy domain="delegate" rights="none" pattern="mpeg:decode" />

  Here we do not want users reading images from HTTP:

    <policy domain="coder" rights="none" pattern="HTTP" />

  The /repository file system is restricted to read o

### Установка ImageMagick
`MoviePy` часто использует `ImageMagick` для обработки текста и некоторых визуальных эффектов. Установим его.

In [ ]:
# Установка ImageMagick
!apt-get install imagemagick
print("ImageMagick установлен.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-droid-fallback fonts-noto-mono fonts-urw-base35 ghostscript gsfonts
  imagemagick-6-common imagemagick-6.q16 libdjvulibre-text libdjvulibre21
  libfftw3-double3 libgs9 libgs9-common libidn12 libijs-0.35 libilmbase25
  libjbig2dec0 libjxr-tools libjxr0 liblqr-1-0 libmagickcore-6.q16-6
  libmagickcore-6.q16-6-extra libmagickwand-6.q16-6 libnetpbm10 libopenexr25
  libwmflite-0.2-7 netpbm poppler-data
Suggested packages:
  fonts-noto fonts-freefont-otf | fonts-freefont-ttf fonts-texgyre
  ghostscript-x imagemagick-doc autotrace cups-bsd | lpr | lprng enscript gimp
  gnuplot grads hp2xx html2ps libwmf-bin mplayer povray radiance sane-utils
  texlive-base-bin transfig ufraw-batch libfftw3-bin libfftw3-dev inkscape
  poppler-utils fonts-japanese-mincho | fonts-ipafont-mincho
  fonts-japanese-gothic | fonts-ipafont-gothic fonts-arphic-uka

### Настройка MoviePy для использования ImageMagick
Теперь, когда ImageMagick установлен, нужно указать MoviePy, где его найти, установив переменную среды `IMAGEMAGICK_BINARY`.

In [ ]:
# Установите путь к исполняемому файлу ImageMagick для MoviePy
import moviepy.config as mpy_config
mpy_config.change_settings({"IMAGEMAGICK_BINARY": "/usr/bin/convert"})
print("MoviePy настроен на использование ImageMagick.")

MoviePy настроен на использование ImageMagick.


Теперь, когда все зависимости установлены и настроены, пожалуйста, запустите основной блок кода (`jURZFdF8RDol`) снова.

In [ ]:
print("\n--- Повторная проверка разделения предложений в original_en.txt ---")
def get_sentences_check(file_path):
    sentences = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                sentences.append(line)
    return sentences

cleaned_sentences_after_fix = get_sentences_check(EN_FILE)

for i, s in enumerate(cleaned_sentences_after_fix):
    print(f"Предложение {i+1}: {s}")
print("--- Повторная проверка завершена ---")


--- Повторная проверка разделения предложений в original_en.txt ---
Предложение 1: The Sun and the North Wind were talking to each other in the sky.
Предложение 2: The North Wind was saying that he was better than everyone else.
Предложение 3: The Sun listened as the North Wind talked with enthusiasm about how powerful he was and how he could push something from one continent to another with one breath.
Предложение 4: He said, “I am the strongest thing in the sky.”
Предложение 5: “Really?” asked the Sun.
Предложение 6: “How do you know that you are more powerful than the stars, or the rain, or even me?”
Предложение 7: The North Wind laughed with disrespect.
Предложение 8: He yelled, “You? That’s a joke!”
Предложение 9: This hurt the Sun.
Предложение 10: He was usually timid and did not want to cause conflict.
Предложение 11: Today he decided that he should teach the North Wind a lesson.
Предложение 12: In the meantime, a man began walking along the avenue down on Earth.
Предложение 13

In [ ]:
print("\n--- Проверка разделения предложений в original_en.txt ---")
with open(EN_FILE, 'r', encoding='utf-8') as f:
    raw_text = f.read()
    processed_text = raw_text.replace('\n', ' ').strip()
    sentences = re.split(r'(?<=[.!?])\s*(?:[”"\)\]])*\s+', processed_text)
    # Filter out empty strings and strip leading/trailing whitespace from each sentence
    cleaned_sentences = [s.strip() for s in sentences if s.strip()]

for i, s in enumerate(cleaned_sentences):
    print(f"Предложение {i+1}: {s}")
print("--- Проверка завершена ---")


--- Проверка разделения предложений в original_en.txt ---
Предложение 1: The Sun and the North Wind were talking to each other in the sky.
Предложение 2: The North Wind was saying that he was better than everyone else.
Предложение 3: The Sun listened as the North Wind talked with enthusiasm about how powerful he was and how he could push something from one continent to another with one breath.
Предложение 4: He said, “I am the strongest thing in the sky.
Предложение 5: “Really?
Предложение 6: asked the Sun.
Предложение 7: “How do you know that you are more powerful than the stars, or the rain, or even me?
Предложение 8: The North Wind laughed with disrespect.
Предложение 9: He yelled, “You?
Предложение 10: That’s a joke!
Предложение 11: This hurt the Sun.
Предложение 12: He was usually timid and did not want to cause conflict.
Предложение 13: Today he decided that he should teach the North Wind a lesson.
Предложение 14: In the meantime, a man began walking along the avenue down on Ear

In [ ]:
import os

en_file_path = 'original_en.txt'

if os.path.exists(en_file_path):
    with open(en_file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        print(f"Content of '{en_file_path}' with line numbers:")
        for i, line in enumerate(lines):
            print(f"{i + 1}: {line.strip()}")
else:
    print(f"Error: File '{en_file_path}' not found.")

Content of 'original_en.txt' with line numbers:
1: The Sun and the North Wind were talking to each other in the sky.
2: The North Wind was saying that he was better than everyone else.
3: The Sun listened as the North Wind talked with enthusiasm about how powerful he was and how he could push something from one continent to another with one breath.
4: He said, "I am the strongest thing in the sky."
5: "Really?" asked the Sun.
6: How do you know that you are more powerful than the stars, or the rain, or even me?
7: The North Wind laughed with disrespect.
8: He yelled, "You? That's a joke!"
9: This hurt the Sun.
10: He was usually timid and did not want to cause conflict.
11: Today he decided that he should teach the North Wind a lesson.
12: In the meantime, a man began walking along the avenue down on Earth.
13: When the Sun looked down on the terrain below, he saw the man.
14: He pointed down to the Earth and said, "Do you see that man walking below?
15: I bet I can get his jacket off 

### Ускоренное Объединение Видео с использованием `ffmpeg` Concat Demuxer

Этот метод использует `ffmpeg` для быстрого объединения всех сгенерированных видеоклипов без их повторного кодирования, что значительно ускоряет процесс и снижает потребление ресурсов. Он особенно полезен для большого количества коротких клипов.


In [ ]:
from moviepy.editor import VideoFileClip, concatenate_videoclips
import os

# --- Объединение видео ---

final_video_output = "final_karaoke_video.mp4"

# Получить список всех сгенерированных видеофайлов
video_files = sorted([os.path.join(OUTPUT_FOLDER, f) for f in os.listdir(OUTPUT_FOLDER) if f.endswith('.mp4')])

if not video_files:
    print(f"Ошибка: В папке '{OUTPUT_FOLDER}' не найдено MP4-видеофайлов для объединения.")
else:
    print(f"Объединяю {len(video_files)} видеофайлов...")
    # Загрузить все видеоклипы
    clips = [VideoFileClip(f) for f in video_files]

    # Объединить их
    final_clip = concatenate_videoclips(clips)

    # Записать финальное видео
    final_clip.write_videofile(final_video_output, fps=24, codec='libx264', audio_codec='aac')

    print(f"Все видеофайлы объединены в '{final_video_output}'.")

    # Опционально: Скачать объединенное видео
    from google.colab import files
    files.download(final_video_output)
    print(f"Файл '{final_video_output}' готов к скачиванию.")

Ошибка: В папке 'simple_karaoke' не найдено MP4-видеофайлов для объединения.


In [ ]:
import pandas as pd
import os
import re # Import the re module for regular expressions

QA_FILE = 'qa_data.csv'
EN_OUTPUT_FILE = 'original_en.txt'
RU_OUTPUT_FILE = 'translated_ru.txt'

if os.path.exists(QA_FILE):
    print(f"Файл '{QA_FILE}' найден. Обрабатываю...")
    try:
        # First attempt: try reading with semicolon as separator.
        # This handles cases where 'question;answer' is not quoted in the header.
        df_qa = pd.read_csv(QA_FILE, sep=';')
        print(f"После pd.read_csv(QA_FILE, sep=';'), столбцы: {df_qa.columns.tolist()}")

        # Normalize column names to lowercase for robust checking
        df_qa.columns = [col.lower() for col in df_qa.columns]

        if 'question' in df_qa.columns and 'answer' in df_qa.columns and len(df_qa.columns) == 2:
            # If successfully read as two distinct columns (case-insensitive)
            print("Столбцы 'question' и 'answer' успешно обнаружены.")
        elif len(df_qa.columns) == 1 and df_qa.columns[0] == 'question;answer':
            # If it's a single column named 'question;answer' (e.g., if the header was quoted like "question;answer")
            print("Обнаружен единственный столбец 'question;answer'. Попытка разделения его содержимого.")
            # Split the content of the single column into two new columns
            # n=1 ensures that only the first semicolon is used for splitting, preventing issues if
            # questions or answers contain semicolons themselves.
            split_data = df_qa[df_qa.columns[0]].str.split(';', n=1, expand=True)

            if split_data.shape[1] == 2:
                df_qa['question'] = split_data[0].str.strip() # Strip whitespace
                df_qa['answer'] = split_data[1].str.strip()   # Strip whitespace
                df_qa = df_qa[['question', 'answer']] # Keep only the newly created columns
                print("Столбцы 'question' и 'answer' успешно созданы путем разделения содержимого.")
            else:
                print(f"Ошибка: Разделение содержимого столбца '{df_qa.columns[0]}' по символу ';' не дало двух частей. Проверьте формат данных в CSV.")
                print(f"Пример данных в столбце '{df_qa.columns[0]}' (первые 5 строк):\n{df_qa[df_qa.columns[0]].head().tolist()}")
                raise ValueError("Не удалось разделить объединенный столбец на 'question' и 'answer'.")
        else:
            # If neither of the above conditions met, it's an unexpected format
            print(f"Ошибка: Неожиданный формат CSV-файла. Ожидались столбцы 'question' и 'answer' или один столбец 'question;answer'. Текущие столбцы: {df_qa.columns.tolist()}")
            raise ValueError("Некорректный формат CSV-файла: отсутствуют ожидаемые столбцы.")

    except Exception as e:
        print(f"Критическая ошибка при чтении или обработке файла '{QA_FILE}': {e}")
        print("Пожалуйста, убедитесь, что ваш файл qa_data.csv корректно отформатирован.")
        exit() # Terminate execution if a critical error occurs

    # Final check after all processing
    if 'question' in df_qa.columns and 'answer' in df_qa.columns:
        print(f"Финальная проверка: Столбцы 'question' и 'answer' присутствуют. Продолжаю.")
        # Записываем ответы в original_en.txt (обратный порядок)
        with open(EN_OUTPUT_FILE, 'w', encoding='utf-8') as f_en:
            for answer in df_qa['answer']:
                # Remove '[number]' before writing to file
                cleaned_answer = re.sub(r'\[\d+\]', '', str(answer)).strip()
                f_en.write(cleaned_answer + '\n')

        # Записываем вопросы в translated_ru.txt (обратный порядок)
        with open(RU_OUTPUT_FILE, 'w', encoding='utf-8') as f_ru:
            for question in df_qa['question']:
                # Remove '[number]' before writing to file
                cleaned_question = re.sub(r'\[\d+\]', '', str(question)).strip()
                f_ru.write(cleaned_question + '\n')

        print(f"Файлы '{EN_OUTPUT_FILE}' и '{RU_OUTPUT_FILE}' успешно созданы из '{QA_FILE}' (содержимое поменялось местами).")
    else:
        # This branch should ideally not be reached if the try-except block is robust
        print(f"Ошибка: Файл '{QA_FILE}' должен содержать столбцы 'question' и 'answer' после обработки. Текущие столбцы: {df_qa.columns.tolist()}")
else:
    print(f"Ошибка: Файл '{QA_FILE}' не найден в текущей директории. Пожалуйста, загрузите его.")

Файл 'qa_data.csv' найден. Обрабатываю...
После pd.read_csv(QA_FILE, sep=';'), столбцы: ['Question', 'Answer']
Столбцы 'question' и 'answer' успешно обнаружены.
Финальная проверка: Столбцы 'question' и 'answer' присутствуют. Продолжаю.
Файлы 'original_en.txt' и 'translated_ru.txt' успешно созданы из 'qa_data.csv' (содержимое поменялось местами).


In [ ]:
import os

en_file_path = 'original_en.txt'

if os.path.exists(en_file_path):
    with open(en_file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        print(f"Content of '{en_file_path}' with line numbers after cleaning:")
        for i, line in enumerate(lines):
            print(f"{i + 1}: {line.strip()}")
else:
    print(f"Error: File '{en_file_path}' not found.")

Content of 'original_en.txt' with line numbers after cleaning:
1: The Sun and the North Wind were talking to each other in the sky.
2: The North Wind was saying that he was better than everyone else.
3: The Sun listened as the North Wind talked with enthusiasm about how powerful he was and how he could push something from one continent to another with one breath.
4: He said, "I am the strongest thing in the sky."
5: Really? asked the Sun.
6: How do you know that you are more powerful than the stars, or the rain, or even me?
7: The North Wind laughed with disrespect.
8: He yelled, "You? That's a joke!"
9: This hurt the Sun.
10: He was usually timid and did not want to cause conflict.
11: Today he decided that he should teach the North Wind a lesson.
12: In the meantime, a man began walking along the avenue down on Earth.
13: When the Sun looked down on the terrain below, he saw the man.
14: He pointed down to the Earth and said, "Do you see that man walking below?
15: I bet I can get hi

# Новый раздел

In [ ]:
!edge-tts --list-voices

Name                               Gender    ContentCategories      VoicePersonalities
---------------------------------  --------  ---------------------  --------------------------------------
af-ZA-AdriNeural                   Female    General                Friendly, Positive
af-ZA-WillemNeural                 Male      General                Friendly, Positive
am-ET-AmehaNeural                  Male      General                Friendly, Positive
am-ET-MekdesNeural                 Female    General                Friendly, Positive
ar-AE-FatimaNeural                 Female    General                Friendly, Positive
ar-AE-HamdanNeural                 Male      General                Friendly, Positive
ar-BH-AliNeural                    Male      General                Friendly, Positive
ar-BH-LailaNeural                  Female    General                Friendly, Positive
ar-DZ-AminaNeural                  Female    General                Friendly, Positive
ar-DZ-IsmaelNeural     

In [ ]:
import asyncio
from IPython.display import Audio, display

# Пример фразы для синтеза
sample_ru_phrase = "Привет, это тестовая фраза на русском языке."
sample_en_phrase = "Hello, this is a test phrase in English."

output_ru_audio = "sample_ru_phrase.mp3"
output_en_audio = "sample_en_phrase.mp3"

async def synthesize_sample_phrases():
    print(f"Синтезирую русскую фразу голосом: {VOICE_RU}")
    await generate_voice(sample_ru_phrase, VOICE_RU, output_ru_audio)
    print(f"Синтезирую английскую фразу голосом: {VOICE_EN}")
    await generate_voice(sample_en_phrase, VOICE_EN, output_en_audio)
    print("Синтез завершен.")

await synthesize_sample_phrases()

print("Прослушать русскую фразу:")
display(Audio(output_ru_audio, autoplay=False))

print("Прослушать английскую фразу:")
display(Audio(output_en_audio, autoplay=False))

Синтезирую русскую фразу голосом: ru-RU-DmitryNeural
Синтезирую английскую фразу голосом: en-US-GuyNeural
Синтез завершен.
Прослушать русскую фразу:


Прослушать английскую фразу:


После того как вы выберете подходящий голос из списка, вы сможете изменить его в следующей ячейке, заменив существующие значения переменных `VOICE_RU` или `VOICE_EN`.

In [ ]:
# Пример изменения голоса (замените на свои выбранные голоса)
VOICE_RU = "ru-RU-SvetlanaNeural"  # Пример: женский голос для русского
VOICE_EN = "en-GB-RyanNeural"      # Пример: британский мужской голос для английского

print(f"Новый русский голос установлен как: {VOICE_RU}")
print(f"Новый английский голос установлен как: {VOICE_EN}")

In [ ]:
import os

output_folder = 'simple_karaoke'
zip_file = 'karaoke_simple.zip'

if os.path.exists(output_folder):
    print(f"Содержимое папки '{output_folder}':")
    for item in os.listdir(output_folder):
        print(f"- {item}")
else:
    print(f"Папка '{output_folder}' не найдена.")

if os.path.exists(zip_file):
    print(f"\nАрхив '{zip_file}' успешно создан.")
else:
    print(f"\nАрхив '{zip_file}' не найден. Возможно, скрипт не завершился до конца.")

In [ ]:
import shutil
import os

folder_to_delete = 'simple_karaoke3' # Замените на имя папки, которую вы хотите удалить

if os.path.exists(folder_to_delete):
    shutil.rmtree(folder_to_delete)
    print(f"Папка '{folder_to_delete}' успешно удалена.")
elif folder_to_delete == 'simple_karaoke3':
    print("Пожалуйста, замените 'simple_karaoke3' на фактическое имя папки, которую вы хотите удалить.")
else:
    print(f"Папка '{folder_to_delete}' не найдена.")


Папка 'simple_karaoke3' успешно удалена.


In [ ]:
await main()

NameError: name 'main' is not defined

In [ ]:
import os

en_file_exists = os.path.exists('original_en.txt')
ru_file_exists = os.path.exists('translated_ru.txt')

if en_file_exists and ru_file_exists:
    print("Файлы 'original_en.txt' и 'translated_ru.txt' успешно загружены.")
elif en_file_exists:
    print("Файл 'original_en.txt' загружен, но 'translated_ru.txt' отсутствует.")
elif ru_file_exists:
    print("Файл 'translated_ru.txt' загружен, но 'original_en.txt' отсутствует.")
else:
    print("Ни один из файлов ('original_en.txt', 'translated_ru.txt') не найден.")

Файлы 'original_en.txt' и 'translated_ru.txt' успешно загружены.


In [ ]:
# Изменение политики ImageMagick для разрешения создания временных файлов
# Эта команда пытается установить все права доступа для всех политик ImageMagick.
# Это должно решить проблему с 'security policy'.
!sudo sed -i 's/rights="none"/rights="all"/g' /etc/ImageMagick-6/policy.xml

print("Политика ImageMagick изменена. Теперь попробуйте повторно запустить основной скрипт.")

Политика ImageMagick изменена. Теперь попробуйте повторно запустить основной скрипт.


In [ ]:
from google.colab import files

print("Пожалуйста, загрузите 'original_en.txt':")
files.upload()

print("Пожалуйста, загрузите 'translated_ru.txt':")
files.upload()

Пожалуйста, загрузите 'original_en.txt':


Saving original_en.txt to original_en (1).txt
Пожалуйста, загрузите 'translated_ru.txt':


Saving translated_ru.txt to translated_ru.txt


{'translated_ru.txt': b'\xd0\x9a\xd1\x82\xd0\xbe \xd1\x80\xd0\xb0\xd0\xb7\xd0\xb3\xd0\xbe\xd0\xb2\xd0\xb0\xd1\x80\xd0\xb8\xd0\xb2\xd0\xb0\xd0\xbb \xd0\xb4\xd1\x80\xd1\x83\xd0\xb3 \xd1\x81 \xd0\xb4\xd1\x80\xd1\x83\xd0\xb3\xd0\xbe\xd0\xbc \xd0\xb2 \xd0\xbd\xd0\xb5\xd0\xb1\xd0\xb5?\r\n\xd0\xa7\xd1\x82\xd0\xbe \xd1\x83\xd1\x82\xd0\xb2\xd0\xb5\xd1\x80\xd0\xb6\xd0\xb4\xd0\xb0\xd0\xbb \xd0\xa1\xd0\xb5\xd0\xb2\xd0\xb5\xd1\x80\xd0\xbd\xd1\x8b\xd0\xb9 \xd0\x92\xd0\xb5\xd1\x82\xd0\xb5\xd1\x80?\r\n\xd0\xa7\xd1\x82\xd0\xbe \xd0\xb4\xd0\xb5\xd0\xbb\xd0\xb0\xd0\xbb\xd0\xbe \xd0\xa1\xd0\xbe\xd0\xbb\xd0\xbd\xd1\x86\xd0\xb5, \xd0\xbf\xd0\xbe\xd0\xba\xd0\xb0 \xd0\xa1\xd0\xb5\xd0\xb2\xd0\xb5\xd1\x80\xd0\xbd\xd1\x8b\xd0\xb9 \xd0\x92\xd0\xb5\xd1\x82\xd0\xb5\xd1\x80 \xd1\x85\xd0\xb2\xd0\xb0\xd1\x81\xd1\x82\xd0\xb0\xd0\xbb\xd1\x81\xd1\x8f \xd1\x81\xd0\xb2\xd0\xbe\xd0\xb5\xd0\xb9 \xd1\x81\xd0\xb8\xd0\xbb\xd0\xbe\xd0\xb9?\r\n\xd0\xa7\xd1\x82\xd0\xbe \xd0\xb7\xd0\xb0\xd1\x8f\xd0\xb2\xd0\xb8\xd0\xbb \xd0\xa1\xd0\